In [ ]:
# Install
!pip install ultralytics

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Update data.yaml paths to Colab paths, then train
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(
    data="/content/drive/MyDrive/merged_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device="0",          # GPU
    patience=20,
    name="smoking_v2"
)


KeyboardInterrupt: 

In [ ]:
from ultralytics.utils.plotting import plot_results;
plot_results('/content/runs/detect/smoking_v2-3/results.csv')

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
from PIL import Image
import io
from ultralytics import YOLO

model = YOLO("/content/runs/detect/smoking_v2-3/weights/best.pt")

COLOURS = {0: (255, 100, 0), 1: (0, 0, 220), 2: (0, 200, 50)}
LABELS  = {0: "Cigarette", 1: "Smoking", 2: "Cig-like Object"}


In [ ]:
def start_webcam():
    js = Javascript('''
        async function startWebcam() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            const canvas = document.createElement('canvas');
            const resultImg = document.createElement('img');

            video.style.display = 'none';
            canvas.style.display = 'none';

            document.body.appendChild(div);
            div.appendChild(video);
            div.appendChild(resultImg);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            canvas.width  = video.videoWidth;
            canvas.height = video.videoHeight;

            window._stopWebcam = false;

            while (!window._stopWebcam) {
                canvas.getContext('2d').drawImage(video, 0, 0);
                const imgData = canvas.toDataURL('image/jpeg', 0.8);
                const result  = await google.colab.kernel.invokeFunction(
                    'notebook.process_frame', [imgData], {}
                );
                resultImg.src = result.data['text/plain'].replace(/^'|'$/g, '');
                await new Promise(r => setTimeout(r, 50));  // ~20fps cap
            }
            stream.getTracks().forEach(t => t.stop());
            div.remove();
        }
        startWebcam();
    ''')
    display(js)

def stop_webcam():
    display(Javascript("window._stopWebcam = true;"))


In [ ]:
import google.colab.output
import base64

CONF_THRESH = 0.4   # adjust as needed

def process_frame(img_data):
    # Decode base64 image from browser
    img_bytes = b64decode(img_data.split(',')[1])
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

    # Run inference
    results = model(frame, conf=CONF_THRESH, verbose=False)[0]

    # Draw boxes
    if results.boxes is not None:
        for box in results.boxes:
            cls_id   = int(box.cls[0])
            conf_val = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            colour = COLOURS.get(cls_id, (200, 200, 200))
            label  = f"{LABELS.get(cls_id, cls_id)} {conf_val:.0%}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+4, y1), colour, -1)
            cv2.putText(frame, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)

    # Encode back to base64 for display
    _, buf = cv2.imencode('.jpg', frame)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf).decode()

google.colab.output.register_callback('notebook.process_frame', process_frame)


In [ ]:
start_webcam()

In [ ]:
stop_webcam()

# **Modifications**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS to wherever your merged_dataset folder is in Drive
DATASET_PATH = "/content/drive/MyDrive/merged_dataset"
RUNS_PATH    = "/content/runs"

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATASET = Path(DATASET_PATH)

def remap_label_file(path: Path):
    """Remap class IDs in a single YOLO label file in-place."""
    lines = path.read_text().strip().splitlines()
    new_lines = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        cls = int(parts[0])

        if cls == 0 or cls == 1:
            new_cls = 0
        elif cls == 2:
            new_cls = 1
        else:
            continue
        new_lines.append(f"{new_cls} " + " ".join(parts[1:]))
    path.write_text("\n".join(new_lines))

total = 0
for split in ["train", "valid", "test"]:
    label_dir = DATASET / split / "labels"
    if not label_dir.exists():
        continue
    files = list(label_dir.glob("*.txt"))
    for f in files:
        remap_label_file(f)
    total += len(files)
    print(f"  {split}: {len(files)} label files remapped")

print(f"\nDone — {total} files remapped")

  train: 5881 label files remapped
  valid: 920 label files remapped
  test: 0 label files remapped

Done — 6801 files remapped


In [ ]:
yaml_content = f"""train: {DATASET_PATH}/train/images
val:   {DATASET_PATH}/valid/images
test:  {DATASET_PATH}/test/images

nc: 2
names:
  - cigarette
  - cigarette_like_object
"""

yaml_path = DATASET / "data.yaml"
yaml_path.write_text(yaml_content)
print(f"data.yaml updated:\n{yaml_content}")

data.yaml updated:
train: /content/drive/MyDrive/merged_dataset/train/images
val:   /content/drive/MyDrive/merged_dataset/valid/images
test:  /content/drive/MyDrive/merged_dataset/test/images

nc: 2
names:
  - cigarette
  - cigarette_like_object



In [ ]:
from collections import Counter

counts = Counter()
for f in (DATASET / "train" / "labels").glob("*.txt"):
    for line in f.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1

print("Train label distribution:")
names = {0: "cigarette", 1: "cigarette_like_object"}
for cls, cnt in sorted(counts.items()):
    print(f"  {names.get(cls, cls)}: {cnt:,} annotations")

Train label distribution:
  cigarette: 7,428 annotations


In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

model = YOLO("yolo11s.pt")

results = model.train(
    data    = str(DATASET / "data.yaml"),
    epochs  = 150,          # more epochs for 2-class — converges cleaner
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    workers = 2,

    # Optimiser
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs = 5,
    cos_lr       = True,    # cosine LR decay — smoother convergence

    # Regularisation
    weight_decay = 0.0005,
    dropout      = 0.0,
    patience     = 30,      # more patience — 2-class trains longer before plateau

    # Augmentation
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    fliplr  = 0.5,
    flipud  = 0.0,
    mosaic  = 1.0,
    mixup   = 0.15,
    erasing = 0.4,

    # Class weights to handle cigarette_like_object having lower recall
    cls = 0.7,

    project  = RUNS_PATH,
    name     = "smoking_v3_2class",
    exist_ok = True,
    verbose  = True,
    cache    = True,
    amp      = True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: NVIDIA L4
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0

In [ ]:
best_model = YOLO(f"{RUNS_PATH}/smoking_v3_2class/weights/best.pt")
metrics = best_model.val(split="test")

print("\n========== TEST SET RESULTS ==========")
print(f"mAP50          : {metrics.box.map50:.4f}")
print(f"mAP50-95       : {metrics.box.map:.4f}")
print(f"Precision      : {metrics.box.p.mean():.4f}")
print(f"Recall         : {metrics.box.r.mean():.4f}")
print("\nPer-class mAP50:")
names = ["cigarette", "cigarette_like_object"]
for i, (name, val) in enumerate(zip(names, metrics.box.maps)):
    print(f"  {name:30s}: {val:.4f}")

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs


FileNotFoundError: [34m[1mval: [0mError loading data from /content/drive/MyDrive/merged_dataset/test/images
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [ ]:
import shutil

save_path = "/content/drive/MyDrive/smoking_v3_best.pt"
shutil.copy(f"{RUNS_PATH}/smoking_v3_2class/weights/best.pt", save_path)
print(f"best.pt saved to Drive: {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  train: 5881 label files remapped
  valid: 920 label files remapped
  test: 0 label files remapped

Done — 6801 files remapped
data.yaml updated:
train: /content/drive/MyDrive/merged_dataset/train/images
val:   /content/drive/MyDrive/merged_dataset/valid/images
test:  /content/drive/MyDrive/merged_dataset/test/images

nc: 2
names:
  - cigarette
  - cigarette_like_object

Train label distribution:
  cigarette: 5,132 annotations
  cigarette_like_object: 2,296 annotations
GPU: Tesla T4
New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.